# texttumor specific MiTRI

In [ ]:
import pandas as pd
import os
from pathlib import Path

# textcancer-type list
cancer_types = ["CRC", "BRCA", "CESC", "LIHC", "OSCC", "LUNG", "PAAD", "STAD", "BLCA"]

# base path
base_path = "/path/to/project/results_V3/cancers_V3.1"

# outputpath
output_dir = "/path/to/project/results_V3/summary/tumor_specific_MTR"
output_file = os.path.join(output_dir, "tumor_specific_MiTRI.csv")

# Create output directory(textexists)
os.makedirs(output_dir, exist_ok=True)

# textalldatalist
all_data = []

# texteachcancer type
for cancer in cancer_types:
    # buildinputfilepath
    input_file = os.path.join(
        base_path,         cancer,         "04.activity",         f"{cancer}_tumor_specific_MiTRI.xlsx"
)
# checkfiletextexists
if os.path.exists(input_file):
    print(f"textprocess: {cancer}")
    # readExcelfile
    df = pd.read_excel(input_file)
    # textCancer_Typecolumn
    df.insert(0, 'Cancer_Type', cancer)
    # texttolist
    all_data.append(df)
    print(f" - readtext {len(df)} rows of data")
else:
    print(f"warning: filetextexists - {input_file}")

# textalldata
if all_data:
    combined_df = pd.concat(all_data, ignore_index=True)
    # saveforCSVfile
    combined_df.to_csv(output_file, index=False)
    print(f"\ntextcompleted!")
    print(f"textprocesstext {len(all_data)} cancer type")
    print(f"total rows: {len(combined_df)}")
    print(f"Output files: {output_file}")
    print(f"\ndatatext:")
    print(combined_df.head())
    print(f"\ntextcancer typedatastatistics:")
    print(combined_df['Cancer_Type'].value_counts())
else:
    print("error: textwithfoundtextdata files!")

# textabundancedata

In [ ]:
import pandas as pd
import os
import gzip

# inputfilepath
input_file = "/path/to/project/results_V3/summary/tumor_specific_MTR/tumor_specific_MiTRI.csv"

# readtextMiTRIdata
print("readtumor_specific_MiTRI.csvfile...")
mitri_df = pd.read_csv(input_file)
print(f"shared {len(mitri_df)} rows of data")

# textpathseq_normalizedcolumn, textforNaN
mitri_df['pathseq_normalized'] = None

# textdatabase path
quant_base_path = "/path/to/project/results_V3/cancers_V3.1"

# statistics
stats = {
    'total': len(mitri_df),
    'matched': 0,
    'not_found_file': 0,
    'not_found_microbe': 0,
    'not_found_sample': 0
}

# textcancer typetextprocess(text)
for cancer_type in mitri_df['Cancer_Type'].unique():
    print(f"\nprocesscancer type: {cancer_type}")
    # buildtextdata filespath
    quant_file = os.path.join(
        quant_base_path,
        cancer_type,
        "01.group_matrix",
        f"{cancer_type}_Tumor_species_pathseq.csv"
)
# checkfiletextexists
if not os.path.exists(quant_file):
    print(f" warning: filetextexists - {quant_file}")
    cancer_rows = mitri_df['Cancer_Type'] == cancer_type
    stats['not_found_file'] += cancer_rows.sum()
    continue
    # readtextdata
    print(f" readtextdata files...")
    quant_df = pd.read_csv(quant_file)
    # textcolumntextfortext(microbetext)
    quant_df = quant_df.set_index(quant_df.columns[0])
    print(f" textdata: {len(quant_df)} microbes, {len(quant_df.columns)} samples")
    # textcancer typealldatarows
    cancer_mask = mitri_df['Cancer_Type'] == cancer_type
    cancer_indices = mitri_df[cancer_mask].index
    # textrowsmatchedtextpathseq_normalizedtext
    for idx in cancer_indices:
        reference = mitri_df.loc[idx, 'Reference']
        sample_name = mitri_df.loc[idx, 'Sample_Name']
        # checkmicrobetextexists
        if reference not in quant_df.index:
            stats['not_found_microbe'] += 1
            continue
        # checksampletextexists
        if sample_name not in quant_df.columns:
            stats['not_found_sample'] += 1
            continue
        # text
        value = quant_df.loc[reference, sample_name]
        mitri_df.loc[idx, 'pathseq_normalized'] = value
        stats['matched'] += 1
        print(f" completedmatched: {stats['matched']} text")

# saveUpdateafterfile
output_file = input_file.replace('.csv', '_with_pathseq.csv')
mitri_df.to_csv(output_file, index=False)

# outputstatistics
print("\n" + "="*60)
print("Processing completed!")
print("="*60)
print(f"textdatarowstext: {stats['total']}")
print(f"successmatched: {stats['matched']} ({stats['matched']/stats['total']*100:.2f}%)")
print(f"filetextexists: {stats['not_found_file']}")
print(f"microbenot found: {stats['not_found_microbe']}")
print(f"samplenot found: {stats['not_found_sample']}")
print(f"\nOutput files: {output_file}")

# textdatatext
print("\ndatatext:")
print(mitri_df.head(10))

# checktextwithtextmatcheddata
unmatched = mitri_df[mitri_df['pathseq_normalized'].isna()]
if len(unmatched) > 0:
    print(f"\nwarning: with {len(unmatched)} rows of datanot foundtextpathseq_normalizedtext")
    print("textmatcheddatatext:")
    print(unmatched.head())
else:
    print("\nOK alldatatextsuccessmatched!")

# textgenetext(gene_repertoires)information

In [ ]:
import pandas as pd
import os
from glob import glob
from collections import defaultdict

# filepath
csv_file = "/path/to/project/results_V3/summary/tumor_specific_MTR/tumor_specific_MiTRI_with_normalized.csv"
diamond_dir = "/path/to/project/data_V3/genes/MiTRI_reads_genes_public/MiTRI_public"
output_file = "/path/to/project/results_V3/summary/tumor_specific_MTR/tumor_specific_MiTRI_with_normalized_gene_diversity.csv"

def extract_microbe_from_qseqid(qseqid):
    """
    fromqseqidinextractmicrobetextlist
    text: "BRCA|Normal|SRR11600329|SRR11600329.10020520|145|Escherichia_coli,Shigella_sonnei|..."
    return: ['Escherichia_coli', 'Shigella_sonnei']
    """
    try:
        parts = qseqid.split('|')
        if len(parts) >= 6:
            microbe_str = parts[5]
            microbes = [m.strip() for m in microbe_str.split(',')]
            return microbes
    except:
        pass
    return []

def find_diamond_file(sample_name, cancer_type, diamond_dir):
    """
    findsampletextdiamondfile
    filenameformat: {Cancer_Type}.{Sample_Type}.{Sample_Name}.diamond_MiTRI.blastx.filtered.txt
    """
    # textmatchedfilenametext
    pattern = os.path.join(diamond_dir, f"{cancer_type}.*.{sample_name}.diamond_MiTRI.blastx.filtered.txt")
    files = glob(pattern)
    if files:
        return files[0]
    # textfound, textmatched
    pattern2 = os.path.join(diamond_dir, f"*.{sample_name}.diamond_MiTRI.blastx.filtered.txt")
    files2 = glob(pattern2)
    if files2:
        return files2[0]
    return None

def count_unique_genes(diamond_file, target_microbe):
    """
    statisticsdiamondfileintextmicrobeuniquegenetext
    """
    if not os.path.exists(diamond_file):
        return 0
    try:
        # readdiamondresultsfile(text)
        df = pd.read_csv(diamond_file, sep='\t', low_memory=False)
        # textrequiredcolumnexists
        if 'qseqid' not in df.columns or 'sseqid' not in df.columns:
            print(f"warning: {diamond_file} textrequiredcolumn")
            return 0
        # filtertexttargetmicroberecord
        matching_rows = []
        for idx, row in df.iterrows():
            qseqid = str(row['qseqid'])
            microbes = extract_microbe_from_qseqid(qseqid)
            # checktargetmicrobetextmicrobe listin
            if any(target_microbe in microbe or microbe in target_microbe for microbe in microbes):
                matching_rows.append(row)
                if not matching_rows:
                    return 0
                # creatematchedrecordDataFrame
                matching_df = pd.DataFrame(matching_rows)
                # statisticsuniquesseqidcount
                unique_genes = matching_df['sseqid'].nunique()
                return unique_genes
    except Exception as e:
        print(f"processfile {diamond_file} text: {str(e)}")
        return 0

# text
def main():
    print("startreadCSVfile...")
    df = pd.read_csv(csv_file, sep=',')
    print(f"readto {len(df)} records")
    print(f"column names: {df.columns.tolist()}")
    # textcolumn
    df['gene_types'] = 0
    # createfiletext, textreadtextfiles
    file_cache = {}
    print("\nstartprocesstextrecords...")
    for idx, row in df.iterrows():
        cancer_type = row['Cancer_Type']
        reference = row['Reference'] # microbetext
        sample_name = row['Sample_Name']
        if (idx + 1) % 100 == 0:
            print(f"processtext: {idx + 1}/{len(df)}")
            # findtextdiamondfile
            diamond_file = find_diamond_file(sample_name, cancer_type, diamond_dir)
            if diamond_file is None:
                print(f"warning: not foundsample {sample_name} (cancer type {cancer_type}) diamondfile")
                continue
            # textfiletextin, textdata
            if diamond_file not in file_cache:
                try:
                    file_cache[diamond_file] = pd.read_csv(diamond_file, sep='\t', low_memory=False)
                except Exception as e:
                    print(f"readfile {diamond_file} failed: {str(e)}")
                    continue
                diamond_df = file_cache[diamond_file]
                # textrequiredcolumnexists
                if 'qseqid' not in diamond_df.columns or 'sseqid' not in diamond_df.columns:
                    continue
                # filtertexttargetmicroberecord
                matching_rows = []
                for _, diamond_row in diamond_df.iterrows():
                    qseqid = str(diamond_row['qseqid'])
                    microbes = extract_microbe_from_qseqid(qseqid)
                    # checktargetmicrobetextmicrobe listin
                    if any(reference in microbe or microbe in reference for microbe in microbes):
                        matching_rows.append(diamond_row['sseqid'])
                        # statisticsuniquegenetext
                        if matching_rows:
                            unique_gene_count = len(set(matching_rows))
                            df.at[idx, 'gene_types'] = unique_gene_count
                            # saveresults
                            print(f"\nsaveresultsto {output_file}...")
                            df.to_csv(output_file, sep=',', index=False)
                            print("completed!")
                            print(f"\nstatistics:")
                            print(f"total records: {len(df)}")
                            print(f"withgenetextdatarecord: {(df['gene_types'] > 0).sum()}")
                            print(f"genetext: {df['gene_types'].min()} - {df['gene_types'].max()}")
                            print(f"averagegenetext: {df['gene_types'].mean():.2f}")

if __name__ == "__main__":
    main()

# textMTRpeptideinformation

## Itext

In [ ]:
import pandas as pd
import os
from glob import glob

# filepath
input_csv = "/path/to/project/results_V3/summary/tumor_specific_MTR/tumor_specific_MiTRI_with_normalized_gene_diversity.csv"
peptide_base_dir = "/path/to/project/results_V3/cancers_V3.1"
output_file = "/path/to/project/results_V3/summary/tumor_specific_MTR/tumor_specific_MiTRI_with_normalized_gene_diversity_peptides.csv"

def find_peptide_file(sample_name, cancer_type, peptide_base_dir):
    """
    findsampletextpeptidefile
    filepathformat: {peptide_base_dir}/{Cancer_Type}/06.antigen/06.2.MTR_peptides/Tumor/{Sample_Name}_classI_filtered.tsv
    """
    peptide_file = os.path.join(
        peptide_base_dir,         cancer_type,         "06.antigen/06.2.MTR_peptides/Tumor",
        f"{sample_name}_classI_filtered.tsv"
)
if os.path.exists(peptide_file):
    return peptide_file
    return None

def count_unique_peptides(peptide_df, target_microbe):
    """
    statisticspeptidefileintextmicrobeuniquepeptide count
    """
    # textrequiredcolumnexists
    if 'peptide' not in peptide_df.columns or 'microbes' not in peptide_df.columns:
        return 0
    # filtertexttargetmicroberecord
    matching_peptides = []
    for _, row in peptide_df.iterrows():
        microbes_str = str(row['microbes'])
        peptide = str(row['peptide'])
        # microbecolumntextmicrobes, text
        microbes = [m.strip() for m in microbes_str.split(',')]
        # checktargetmicrobetextmicrobe listin
        # textmatched: target_microbe in microbe or microbe in target_microbe
        if any(target_microbe in microbe or microbe in target_microbe for microbe in microbes):
            matching_peptides.append(peptide)
            # statisticsuniquepeptide count
            unique_peptides = len(set(matching_peptides))
            return unique_peptides

# text
def main():
    print("startreadCSVfile...")
    df = pd.read_csv(input_csv, sep=',')
    print(f"readto {len(df)} records")
    print(f"column names: {df.columns.tolist()}")
    # textcolumn
    df['peptide_types'] = 0
    # createfiletext, textreadtextfiles
    file_cache = {}
    print("\nstartprocesstextrecords...")
    for idx, row in df.iterrows():
        cancer_type = row['Cancer_Type']
        reference = row['Reference'] # microbetext
        sample_name = row['Sample_Name']
        if (idx + 1) % 100 == 0:
            print(f"processtext: {idx + 1}/{len(df)}")
            # findtextpeptidefile
            peptide_file = find_peptide_file(sample_name, cancer_type, peptide_base_dir)
            if peptide_file is None:
                if (idx + 1) % 500 == 0: # textwarningoutputtext
                    print(f"warning: not foundsample {sample_name} (cancer type {cancer_type}) peptidefile")
                    continue
                # textfiletextin, textdata
                if peptide_file not in file_cache:
                    try:
                        file_cache[peptide_file] = pd.read_csv(peptide_file, sep='\t', low_memory=False)
                    except Exception as e:
                        print(f"readfile {peptide_file} failed: {str(e)}")
                        continue
                    peptide_df = file_cache[peptide_file]
                    # statisticsuniquepeptide count
                    unique_peptide_count = count_unique_peptides(peptide_df, reference)
                    df.at[idx, 'peptide_types'] = unique_peptide_count
                    # saveresults
                    print(f"\nsaveresultsto {output_file}...")
                    df.to_csv(output_file, sep=',', index=False)
                    print("completed!")
                    print(f"\nstatistics:")
                    print(f"total records: {len(df)}")
                    print(f"withpeptidedatarecord: {(df['peptide_types'] > 0).sum()}")
                    print(f"peptidetext: {df['peptide_types'].min()} - {df['peptide_types'].max()}")
                    print(f"averagepeptidetext: {df['peptide_types'].mean():.2f}")
                    # textpeptide_typestext
                    print(f"\npeptide_typestext:")
                    print(df['peptide_types'].value_counts().sort_index().head(20))

if __name__ == "__main__":
    main()

## IItext

In [ ]:
import pandas as pd
import os
from glob import glob

# filepath
input_csv = "/path/to/project/results_V3/summary/tumor_specific_MTR/tumor_specific_MiTRI_with_normalized_gene_diversity_peptides_immunoTop5.csv"
peptide_base_dir = "/path/to/project/results_V3/cancers_V3.1"
output_file = "/path/to/project/results_V3/summary/tumor_specific_MTR/tumor_specific_MiTRI_with_normalized_gene_diversity_peptides_immunoTop5_I_II.csv"

def find_peptide_file(sample_name, cancer_type, peptide_base_dir, peptide_class):
    """
    findsampletextpeptidefile
    peptide_class: 'classI' or 'classII'
    filepathformat: {peptide_base_dir}/{Cancer_Type}/06.antigen/06.2.MTR_peptides/Tumor/{Sample_Name}_{class}_filtered.tsv
    """
    peptide_file = os.path.join(
        peptide_base_dir,         cancer_type,         "06.antigen/06.2.MTR_peptides/Tumor",
        f"{sample_name}_{peptide_class}_filtered.tsv"
)
if os.path.exists(peptide_file):
    return peptide_file
    return None

def count_unique_peptides(peptide_df, target_microbe):
    """
    statisticspeptidefileintextmicrobeuniquepeptide count
    """
    # textrequiredcolumnexists
    if 'peptide' not in peptide_df.columns or 'microbes' not in peptide_df.columns:
        return 0
    # filtertexttargetmicroberecord
    matching_peptides = []
    for _, row in peptide_df.iterrows():
        microbes_str = str(row['microbes'])
        peptide = str(row['peptide'])
        # microbecolumntextmicrobes, text
        microbes = [m.strip() for m in microbes_str.split(',')]
        # checktargetmicrobetextmicrobe listin
        # textmatched: target_microbe in microbe or microbe in target_microbe
        if any(target_microbe in microbe or microbe in target_microbe for microbe in microbes):
            matching_peptides.append(peptide)
            # statisticsuniquepeptide count
            unique_peptides = len(set(matching_peptides))
            return unique_peptides

# text
def main():
    print("startreadCSVfile...")
    df = pd.read_csv(input_csv, sep=',')
    print(f"readto {len(df)} records")
    print(f"column names: {df.columns.tolist()}")
    # textcolumn
    df['peptide_types_classI'] = 0
    df['peptide_types_classII'] = 0
    df['peptide_types_total'] = 0
    # createfiletext, textreadtextfiles
    file_cache = {}
    print("\nstartprocesstextrecords...")
    for idx, row in df.iterrows():
        cancer_type = row['Cancer_Type']
        reference = row['Reference'] # microbetext
        sample_name = row['Sample_Name']
        if (idx + 1) % 100 == 0:
            print(f"processtext: {idx + 1}/{len(df)}")
            # process Class I peptide
            peptide_file_I = find_peptide_file(sample_name, cancer_type, peptide_base_dir, 'classI')
            unique_peptide_count_I = 0
            if peptide_file_I is not None:
                # textfiletextin, textdata
                if peptide_file_I not in file_cache:
                    try:
                        file_cache[peptide_file_I] = pd.read_csv(peptide_file_I, sep='\t', low_memory=False)
                    except Exception as e:
                        print(f"readfile {peptide_file_I} failed: {str(e)}")
                        if peptide_file_I in file_cache:
                            peptide_df_I = file_cache[peptide_file_I]
                            unique_peptide_count_I = count_unique_peptides(peptide_df_I, reference)
                            df.at[idx, 'peptide_types_classI'] = unique_peptide_count_I
                            # process Class II peptide
                            peptide_file_II = find_peptide_file(sample_name, cancer_type, peptide_base_dir, 'classII')
                            unique_peptide_count_II = 0
                            if peptide_file_II is not None:
                                # textfiletextin, textdata
                                if peptide_file_II not in file_cache:
                                    try:
                                        file_cache[peptide_file_II] = pd.read_csv(peptide_file_II, sep='\t', low_memory=False)
                                    except Exception as e:
                                        print(f"readfile {peptide_file_II} failed: {str(e)}")
                                        if peptide_file_II in file_cache:
                                            peptide_df_II = file_cache[peptide_file_II]
                                            unique_peptide_count_II = count_unique_peptides(peptide_df_II, reference)
                                            df.at[idx, 'peptide_types_classII'] = unique_peptide_count_II
                                            # calculatetext
                                            df.at[idx, 'peptide_types_total'] = unique_peptide_count_I + unique_peptide_count_II
                                            # warninginformation(textoutputtext)
                                            if peptide_file_I is None and peptide_file_II is None and (idx + 1) % 500 == 0:
                                                print(f"warning: not foundsample {sample_name} (cancer type {cancer_type}) textpeptidefile")
                                                # saveresults
                                                print(f"\nsaveresultsto {output_file}...")
                                                df.to_csv(output_file, sep=',', index=False)
                                                print("completed!")
                                                print(f"\nstatistics:")
                                                print(f"total records: {len(df)}")
                                                print(f"withClass I peptidedatarecord: {(df['peptide_types_classI'] > 0).sum()}")
                                                print(f"withClass II peptidedatarecord: {(df['peptide_types_classII'] > 0).sum()}")
                                                print(f"withtextpeptidedatarecord: {(df['peptide_types_total'] > 0).sum()}")
                                                print(f"\nClass I peptidetext:")
                                                print(f" text: {df['peptide_types_classI'].min()} - {df['peptide_types_classI'].max()}")
                                                print(f" average: {df['peptide_types_classI'].mean():.2f}")
                                                print(f"\nClass II peptidetext:")
                                                print(f" text: {df['peptide_types_classII'].min()} - {df['peptide_types_classII'].max()}")
                                                print(f" average: {df['peptide_types_classII'].mean():.2f}")
                                                print(f"\ntextpeptidetext:")
                                                print(f" text: {df['peptide_types_total'].min()} - {df['peptide_types_total'].max()}")
                                                print(f" average: {df['peptide_types_total'].mean():.2f}")
                                                # textpeptide_typestext
                                                print(f"\npeptide_types_totaltext(first20):")
                                                print(df['peptide_types_total'].value_counts().sort_index().head(20))

if __name__ == "__main__":
    main()

# textimm

In [ ]:
import pandas as pd
import numpy as np
import os

# ====== path ======
mitri_path = "/path/to/project/results_V3/summary/tumor_specific_MTR/tumor_specific_MiTRI_with_normalized_gene_diversity_peptides.csv"
immuno_path = "/path/to/project/results_V3/summary/MTR_peptide_imm/Imm_peptide_with_immunogenicity.csv"

out_dir = "/path/to/project/results_V3/summary/tumor_specific_MTR/"
out_prefix = "tumor_specific_MiTRI_with_normalized_gene_diversity_peptides_immuno"

# text k list(4 Top-k)
k_list = [1, 3, 5, 10]

# ====== text ======
mitri = pd.read_csv(mitri_path)
imm = pd.read_csv(immuno_path)

# ====== textcolumn names(text/text)======
imm = imm.copy()
imm["Sample_Name"] = imm["sample"]

# microbes text "A|B|C" -> explode
imm["Reference"] = imm["microbes"].astype(str).str.split("|")
imm = imm.explode("Reference")
imm["Reference"] = imm["Reference"].astype(str).str.strip()

# ====== textcolumntext ======
score_cols = ["IEDB_score", "DeepImmuno_score", "BigMHC_IM_score"]
for c in score_cols:
    if c in imm.columns:
        imm[c] = pd.to_numeric(imm[c], errors="coerce")

# ====== Step 1: texteach (Sample_Name, Reference, peptide) text"textHLA"(max) ======
peptide_best = (
    imm.groupby(["Sample_Name", "Reference", "peptide"], as_index=False)[score_cols]
    .max()
)

# ====== Top-k mean text ======
def topk_mean(x: pd.Series, k: int):
    x = x.dropna()
    if x.empty:
        return np.nan
    return x.nlargest(min(k, len(x))).mean()

# ====== generate Top-k file(Top1/3/5/10) ======
os.makedirs(out_dir, exist_ok=True)

for k in k_list:
    agg = (
        peptide_best.groupby(["Sample_Name", "Reference"], as_index=False)
        .agg(
        **{
        f"IEDB_top{k}mean": ("IEDB_score", lambda s, kk=k: topk_mean(s, kk)),
        f"DeepImmuno_top{k}mean": ("DeepImmuno_score", lambda s, kk=k: topk_mean(s, kk)),
        f"BigMHC_top{k}mean": ("BigMHC_IM_score", lambda s, kk=k: topk_mean(s, kk)),
        "n_peptides_immuno": ("peptide", "nunique"),
    }
)
)

mitri_merged = mitri.merge(
    agg,
    on=["Sample_Name", "Reference"],
    how="left",
    validate="m:1"
)

out_path = os.path.join(out_dir, f"{out_prefix}Top{k}.csv")
mitri_merged.to_csv(out_path, index=False)

print(f"OK textoutput: {out_path}")
print("newcolumn: ", [f"IEDB_top{k}mean", f"DeepImmuno_top{k}mean", f"BigMHC_top{k}mean", "n_peptides_immuno"])

# ====== (text5files)TopAll: textall peptide(each peptide text best HLA)text mean ======
# textfortext baseline(optional, text)
agg_all = (
    peptide_best.groupby(["Sample_Name", "Reference"], as_index=False)
    .agg(
    IEDB_allmean=("IEDB_score", lambda s: s.dropna().mean() if s.dropna().size > 0 else np.nan),
    DeepImmuno_allmean=("DeepImmuno_score", lambda s: s.dropna().mean() if s.dropna().size > 0 else np.nan),
    BigMHC_allmean=("BigMHC_IM_score", lambda s: s.dropna().mean() if s.dropna().size > 0 else np.nan),
    n_peptides_immuno=("peptide", "nunique"),
)
)

mitri_merged_all = mitri.merge(
    agg_all,
    on=["Sample_Name", "Reference"],
    how="left",
    validate="m:1"
)

out_path_all = os.path.join(out_dir, f"{out_prefix}All.csv")
mitri_merged_all.to_csv(out_path_all, index=False)
print(f"OK textoutput: {out_path_all}")
print("newcolumn: ", ["IEDB_allmean", "DeepImmuno_allmean", "BigMHC_allmean", "n_peptides_immuno"])
